# Lab 3 · Build the **Gold** `resident_360` table

One row per resident, harmonising every domain:
**Activity + Screening** (Databricks mirror) + **Diet + Events + Programmes + Rewards + eVouchers + Challenges** (Fabric silver).

> **Attach** the `lh_resident360` Lakehouse first.
> Set `DBX` below to your Lab 1 mirror name if it differs from `hpb_databricks_mirror`.

In [ ]:
from pyspark.sql import functions as F
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
DBX = "hpb_databricks_mirror.gold"     # your mirrored Databricks catalog.schema

## 1. Demographics + activity + screening — from the mirrored Databricks estate

In [ ]:
dim = spark.table(f"{DBX}.dim_resident")

act = (spark.table(f"{DBX}.daily_activity").groupBy("resident_id").agg(
        F.avg("steps").alias("avg_daily_steps"),
        F.avg("mvpa_minutes").alias("avg_mvpa_min"),
        F.avg("sleep_minutes").alias("avg_sleep_min"),
        F.sum("goal_met").alias("days_goal_met"),
        F.count("*").alias("active_days")))

# latest screening per resident (most recent screening_date)
from pyspark.sql import Window
w = Window.partitionBy("resident_id").orderBy(F.col("screening_date").desc())
scr = (spark.table(f"{DBX}.health_screening")
       .withColumn("rn", F.row_number().over(w)).filter("rn = 1")
       .select("resident_id", F.col("bmi").alias("latest_bmi"),
               F.col("systolic_bp").alias("latest_systolic"),
               F.col("risk_band").alias("screening_risk")))

## 2. Fabric-native domains — from silver

In [ ]:
meal = (spark.table("silver.fact_meal_log").groupBy("resident_id").agg(
            F.count("*").alias("meal_logs"),
            F.avg("calories").alias("avg_calories"),
            F.avg(F.when(F.col("healthier_choice_flag") == "Y", 1).otherwise(0)).alias("pct_healthier_choice")))

evt = (spark.table("silver.fact_event_attendance").groupBy("resident_id").agg(
            F.sum(F.when(F.col("attended_flag") == "Y", 1).otherwise(0)).alias("events_attended"),
            F.count("*").alias("events_booked")))

prog = (spark.table("silver.fact_programme_enrolment").groupBy("resident_id").agg(
            F.count("*").alias("programmes_enrolled"),
            F.sum(F.when(F.col("status") == "Dropped", 1).otherwise(0)).alias("programmes_dropped")))

rew = (spark.table("silver.fact_rewards").groupBy("resident_id").agg(
            F.sum("points_earned").alias("healthpoints_earned"),
            F.sum("points_redeemed").alias("healthpoints_redeemed")))

vou = (spark.table("silver.fact_evoucher_redemption").groupBy("resident_id").agg(
            F.count("*").alias("vouchers_redeemed"),
            F.sum("voucher_value_sgd").alias("voucher_value_sgd")))

cha = (spark.table("silver.fact_challenge").groupBy("resident_id").agg(
            F.sum(F.when(F.col("status") == "Active", 1).otherwise(0)).alias("challenges_active"),
            F.avg("progress_pct").alias("avg_challenge_progress")))

## 2b. Regional environmental context — air quality (PSI)
Attach each resident's **regional air quality** so the 360 can explain *why* Rahim skips outdoor events on hazy days. Uses `bronze.env_air_quality` from Lab 2 (falls back to safe values if you skipped that notebook).

In [ ]:
try:
    airq = (spark.table("bronze.env_air_quality")
                 .groupBy("region").agg(F.round(F.avg("psi_24h")).cast("int").alias("region_psi")))
    assert airq.count() > 0
    print("air quality: from bronze.env_air_quality")
except Exception as e:
    print("bronze.env_air_quality unavailable — using fallback PSI:", str(e)[:80])
    airq = spark.createDataFrame(
        [("West", 55), ("East", 48), ("Central", 52), ("North", 60), ("North-East", 50)],
        ["region", "region_psi"])
airq.show()

## 3. Join into one Resident 360 row + derive `is_disengaged`
`is_disengaged = 1` when avg steps < 4000 **and** no events attended **and** a dropped programme.
We also attach `region_psi` and a `region_is_hazy` flag (24-hour PSI ≥ 55) for the environmental view.

In [ ]:
r360 = (dim.join(act, "resident_id", "left").join(scr, "resident_id", "left")
           .join(meal, "resident_id", "left").join(evt, "resident_id", "left")
           .join(prog, "resident_id", "left").join(rew, "resident_id", "left")
           .join(vou, "resident_id", "left").join(cha, "resident_id", "left")
           .join(airq, "region", "left")
           .fillna({"events_attended": 0, "programmes_dropped": 0, "avg_daily_steps": 0,
                    "vouchers_redeemed": 0, "healthpoints_earned": 0})
           .withColumn("screening_risk", F.coalesce("screening_risk", F.lit("Not Screened")))
           .withColumn("region_psi", F.coalesce("region_psi", F.lit(50)))
           .withColumn("region_is_hazy", F.when(F.col("region_psi") >= 55, 1).otherwise(0))
           .withColumn("is_disengaged",
               F.when((F.col("avg_daily_steps") < 4000) & (F.col("events_attended") < 1)
                      & (F.col("programmes_dropped") > 0), 1).otherwise(0)))

r360.write.mode("overwrite").option("mergeSchema", True).saveAsTable("gold.resident_360")
print("gold.resident_360:", spark.table("gold.resident_360").count(), "rows")

## 4. Verify — profile + disengagement split

In [ ]:
g = spark.table("gold.resident_360")
print("columns:", len(g.columns))
g.groupBy("is_disengaged").count().show()
display(g.select("resident_id","region","region_is_hazy","age_band","avg_daily_steps","events_attended",
                 "programmes_dropped","screening_risk","healthpoints_earned","is_disengaged").limit(20))